In [5]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [6]:
ratings = pd.read_csv("../data/raw/u.data", sep="\t", names=["user_id","movie_id","rating","timestamp"])

In [7]:
print(ratings.head())

   user_id  movie_id  rating  timestamp
0      196       242       3  881250949
1      186       302       3  891717742
2       22       377       1  878887116
3      244        51       2  880606923
4      166       346       1  886397596


In [8]:
ratings.shape

(100000, 4)

In [9]:
ratings.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   user_id    100000 non-null  int64
 1   movie_id   100000 non-null  int64
 2   rating     100000 non-null  int64
 3   timestamp  100000 non-null  int64
dtypes: int64(4)
memory usage: 3.1 MB


In [10]:
ratings.describe()

,user_id,movie_id,rating,timestamp
count,100000.00000,100000.000000,100000.000000,1.000000e+05
mean,462.48475,425.530130,3.529860,8.835289e+08
std,266.61442,330.798356,1.125674,5.343856e+06
min,1.00000,1.000000,1.000000,8.747247e+08
25%,254.00000,175.000000,3.000000,8.794487e+08
50%,447.00000,322.000000,4.000000,8.828269e+08
75%,682.00000,631.000000,4.000000,8.882600e+08
max,943.00000,1682.000000,5.000000,8.932866e+08


In [11]:
print(ratings.isnull().sum())


user_id      0
movie_id     0
rating       0
timestamp    0
dtype: int64


In [12]:
print("Number of users:", ratings["user_id"].nunique())
print("Number of movies:", ratings["movie_id"].nunique())
print("Average rating:", ratings["rating"].mean())
print("Minimum rating:", ratings["rating"].min())
print("Maximum rating:", ratings["rating"].max())


Number of users: 943
Number of movies: 1682
Average rating: 3.52986
Minimum rating: 1
Maximum rating: 5


In [13]:
ratings["user_id"].nunique()

943

In [14]:
ratings["movie_id"].nunique()

1682

In [15]:
ratings["rating"].value_counts().sort_index()

rating
1     6110
2    11370
3    27145
4    34174
5    21201
Name: count, dtype: int64

Finding most rated movies

In [16]:
movies = pd.read_csv("../data/raw/u.item",sep="|", encoding="latin-1",header=None)

In [17]:
movies.head()

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [18]:
movie_columns = [
    "movie_id",
    "movie_title",
    "release_date",
    "video_release_date",
    "imdb_url",
    "unknown",
    "Action",
    "Adventure",
    "Animation",
    "Children",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Fantasy",
    "Film-Noir",
    "Horror",
    "Musical",
    "Mystery",
    "Romance",
    "Sci-Fi",
    "Thriller",
    "War",
    "Western"
]

movies.columns = movie_columns


In [19]:
movies.head()

,movie_id,movie_title,release_date,video_release_date,imdb_url,unknown,Action,Adventure,Animation,Children,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [20]:
movies.shape

(1682, 24)

In [21]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 1682 entries, 0 to 1681
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   movie_id            1682 non-null   int64  
 1   movie_title         1682 non-null   str    
 2   release_date        1681 non-null   str    
 3   video_release_date  0 non-null      float64
 4   imdb_url            1679 non-null   str    
 5   unknown             1682 non-null   int64  
 6   Action              1682 non-null   int64  
 7   Adventure           1682 non-null   int64  
 8   Animation           1682 non-null   int64  
 9   Children            1682 non-null   int64  
 10  Comedy              1682 non-null   int64  
 11  Crime               1682 non-null   int64  
 12  Documentary         1682 non-null   int64  
 13  Drama               1682 non-null   int64  
 14  Fantasy             1682 non-null   int64  
 15  Film-Noir           1682 non-null   int64  
 16  Horror           

In [22]:
movies.describe()

,movie_id,video_release_date,unknown,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
count,1682.000000,0.0,1682.000000,1682.000000,1682.000000,1682.000000,1682.000000,1682.000000,1682.000000,1682.000000,...,1682.00000,1682.000000,1682.000000,1682.000000,1682.000000,1682.000000,1682.000000,1682.000000,1682.000000,1682.000000
mean,841.500000,NaN,0.001189,0.149227,0.080262,0.024970,0.072533,0.300238,0.064804,0.029727,...,0.01308,0.014269,0.054697,0.033294,0.036266,0.146849,0.060048,0.149227,0.042212,0.016052
std,485.695893,NaN,0.034473,0.356418,0.271779,0.156081,0.259445,0.458498,0.246253,0.169882,...,0.11365,0.118632,0.227455,0.179456,0.187008,0.354061,0.237646,0.356418,0.201131,0.125714
min,1.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,421.250000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,841.500000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1261.750000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1682.000000,NaN,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [23]:
ratings_movies = ratings.merge(movies,on="movie_id")

In [24]:
ratings_movies.head()

,user_id,movie_id,rating,timestamp,movie_title,release_date,video_release_date,imdb_url,unknown,Action,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,196,242,3,881250949,Kolya (1996),24-Jan-1997,NaN,http://us.imdb.com/M/title-exact?Kolya%20(1996),0,0,...,0,0,0,0,0,0,0,0,0,0
1,186,302,3,891717742,L.A. Confidential (1997),01-Jan-1997,NaN,http://us.imdb.com/M/title-exact?L%2EA%2E+Conf...,0,0,...,0,1,0,0,1,0,0,1,0,0
2,22,377,1,878887116,Heavyweights (1994),01-Jan-1994,NaN,http://us.imdb.com/M/title-exact?Heavyweights%...,0,0,...,0,0,0,0,0,0,0,0,0,0
3,244,51,2,880606923,Legends of the Fall (1994),01-Jan-1994,NaN,http://us.imdb.com/M/title-exact?Legends%20of%...,0,0,...,0,0,0,0,0,1,0,0,1,1
4,166,346,1,886397596,Jackie Brown (1997),01-Jan-1997,NaN,http://us.imdb.com/M/title-exact?imdb-title-11...,0,0,...,0,0,0,0,0,0,0,0,0,0


In [25]:
print(ratings_movies.shape)


(100000, 27)


In [26]:
ratings_movies.head()


,user_id,movie_id,rating,timestamp,movie_title,release_date,video_release_date,imdb_url,unknown,Action,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,196,242,3,881250949,Kolya (1996),24-Jan-1997,NaN,http://us.imdb.com/M/title-exact?Kolya%20(1996),0,0,...,0,0,0,0,0,0,0,0,0,0
1,186,302,3,891717742,L.A. Confidential (1997),01-Jan-1997,NaN,http://us.imdb.com/M/title-exact?L%2EA%2E+Conf...,0,0,...,0,1,0,0,1,0,0,1,0,0
2,22,377,1,878887116,Heavyweights (1994),01-Jan-1994,NaN,http://us.imdb.com/M/title-exact?Heavyweights%...,0,0,...,0,0,0,0,0,0,0,0,0,0
3,244,51,2,880606923,Legends of the Fall (1994),01-Jan-1994,NaN,http://us.imdb.com/M/title-exact?Legends%20of%...,0,0,...,0,0,0,0,0,1,0,0,1,1
4,166,346,1,886397596,Jackie Brown (1997),01-Jan-1997,NaN,http://us.imdb.com/M/title-exact?imdb-title-11...,0,0,...,0,0,0,0,0,0,0,0,0,0


In [27]:
ratings_movies.info()


<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 27 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   user_id             100000 non-null  int64  
 1   movie_id            100000 non-null  int64  
 2   rating              100000 non-null  int64  
 3   timestamp           100000 non-null  int64  
 4   movie_title         100000 non-null  str    
 5   release_date        99991 non-null   str    
 6   video_release_date  0 non-null       float64
 7   imdb_url            99987 non-null   str    
 8   unknown             100000 non-null  int64  
 9   Action              100000 non-null  int64  
 10  Adventure           100000 non-null  int64  
 11  Animation           100000 non-null  int64  
 12  Children            100000 non-null  int64  
 13  Comedy              100000 non-null  int64  
 14  Crime               100000 non-null  int64  
 15  Documentary         100000 non-null  int64  
 

In [28]:
most_rated = ratings_movies.groupby("movie_title").size().sort_values(ascending=False)

In [29]:
print(most_rated.head(10))

movie_title
Star Wars (1977)                 583
Contact (1997)                   509
Fargo (1996)                     508
Return of the Jedi (1983)        507
Liar Liar (1997)                 485
English Patient, The (1996)      481
Scream (1996)                    478
Toy Story (1995)                 452
Air Force One (1997)             431
Independence Day (ID4) (1996)    429
dtype: int64


In [30]:
ratings_movies = ratings.merge(
    movies,
    on="movie_id"
)

print(ratings_movies.shape)

print(ratings_movies.head())


(100000, 27)
   user_id  movie_id  rating  timestamp                 movie_title  \
0      196       242       3  881250949                Kolya (1996)   
1      186       302       3  891717742    L.A. Confidential (1997)   
2       22       377       1  878887116         Heavyweights (1994)   
3      244        51       2  880606923  Legends of the Fall (1994)   
4      166       346       1  886397596         Jackie Brown (1997)   

  release_date  video_release_date  \
0  24-Jan-1997                 NaN   
1  01-Jan-1997                 NaN   
2  01-Jan-1994                 NaN   
3  01-Jan-1994                 NaN   
4  01-Jan-1997                 NaN   

                                            imdb_url  unknown  Action  ...  \
0    http://us.imdb.com/M/title-exact?Kolya%20(1996)        0       0  ...   
1  http://us.imdb.com/M/title-exact?L%2EA%2E+Conf...        0       0  ...   
2  http://us.imdb.com/M/title-exact?Heavyweights%...        0       0  ...   
3  http://us.imdb.c

In [31]:
most_rated = (
    ratings_movies
    .groupby("movie_title")
    .size()
    .sort_values(ascending=False)
)

print(most_rated.head(10))


movie_title
Star Wars (1977)                 583
Contact (1997)                   509
Fargo (1996)                     508
Return of the Jedi (1983)        507
Liar Liar (1997)                 485
English Patient, The (1996)      481
Scream (1996)                    478
Toy Story (1995)                 452
Air Force One (1997)             431
Independence Day (ID4) (1996)    429
dtype: int64


In [32]:
global_mean = ratings["rating"].mean()

movie_avg = ratings.groupby("movie_id")["rating"].mean()
print("Global mean rating:", global_mean)
print(movie_avg.head())

Global mean rating: 3.52986
movie_id
1    3.878319
2    3.206107
3    3.033333
4    3.550239
5    3.302326
Name: rating, dtype: float64


User-Item Matrix

In [33]:
user_item_matrix = ratings.pivot_table(
    index="user_id",columns="movie_id",values="rating")


print(user_item_matrix.shape)
user_item_matrix.head()

(943, 1682)


movie_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
from sklearn.metrics.pairwise import cosine_similarity

# fill NaN with 0 for similarity computation only
item_matrix_filled = user_item_matrix.fillna(0)

item_similarity = cosine_similarity(item_matrix_filled.T)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

item_similarity_df.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
movie_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.402382,0.330245,0.454938,0.286714,0.116344,0.620979,0.481114,0.496288,0.273935,...,0.035387,0.0,0.000000,0.000000,0.035387,0.0,0.0,0.0,0.047183,0.047183
2,0.402382,1.000000,0.273069,0.502571,0.318836,0.083563,0.383403,0.337002,0.255252,0.171082,...,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.078299,0.078299
3,0.330245,0.273069,1.000000,0.324866,0.212957,0.106722,0.372921,0.200794,0.273669,0.158104,...,0.000000,0.0,0.000000,0.000000,0.032292,0.0,0.0,0.0,0.000000,0.096875
4,0.454938,0.502571,0.324866,1.000000,0.334239,0.090308,0.489283,0.490236,0.419044,0.252561,...,0.000000,0.0,0.094022,0.094022,0.037609,0.0,0.0,0.0,0.056413,0.075218
5,0.286714,0.318836,0.212957,0.334239,1.000000,0.037299,0.334769,0.259161,0.272448,0.055453,...,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.094211


In [35]:
def predict_rating(user_id, movie_id, k=20):
    if movie_id not in item_similarity_df.columns:
        return global_mean

    user_ratings = user_item_matrix.loc[user_id].dropna()

    if user_ratings.empty:
        return global_mean

    sims = item_similarity_df[movie_id][user_ratings.index]

    # keep only the k most similar movies the user has rated
    top_k = sims.sort_values(ascending=False).head(k)
    top_k = top_k[top_k > 0]

    if top_k.empty:
        return movie_avg.get(movie_id, global_mean)

    weighted_sum = (top_k * user_ratings[top_k.index]).sum()
    sim_sum = top_k.sum()

    return weighted_sum / sim_sum

# quick sanity check
print(predict_rating(196, 242))

3.506119576993453


In [36]:
train = pd.read_csv("../data/raw/u1.base", sep="\t", names=["user_id","movie_id","rating","timestamp"])
test = pd.read_csv("../data/raw/u1.test", sep="\t", names=["user_id","movie_id","rating","timestamp"])

# rebuild the matrix and similarity using ONLY train data
user_item_matrix = train.pivot_table(index="user_id", columns="movie_id", values="rating")
item_matrix_filled = user_item_matrix.fillna(0)
item_similarity = cosine_similarity(item_matrix_filled.T)
item_similarity_df = pd.DataFrame(item_similarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)
movie_avg = train.groupby("movie_id")["rating"].mean()
global_mean = train["rating"].mean()

import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

test["predicted"] = test.apply(
    lambda row: predict_rating(row["user_id"], row["movie_id"]),
    axis=1
)

rmse = np.sqrt(mean_squared_error(test["rating"], test["predicted"]))
mae = mean_absolute_error(test["rating"], test["predicted"])

print("RMSE:", rmse)
print("MAE:", mae)

# compare to a naive baseline: always predict global mean
baseline_rmse = np.sqrt(mean_squared_error(test["rating"], [global_mean]*len(test)))
print("Baseline RMSE (global mean):", baseline_rmse)

RMSE: 0.9975778309621527
MAE: 0.7772768818839199
Baseline RMSE (global mean): 1.1536759477860323


In [38]:
rating_counts=train.groupby("movie_id").size()
popular_movies = rating_counts[rating_counts>=20].index


def recommend_movies(user_id,n=10,min_ratings=20):
    user_rated=user_item_matrix.loc[user_id].dropna().index
    all_movies = [m for m in user_item_matrix.columns if m in popular_movies]
    unrated =[m for m in all_movies if m  not in user_rated]
    
    predictions = [(m,predict_rating(user_id,m))for m in unrated]
    predictions.sort(key=lambda x:[1],reverse=True)
    
    top_n = predictions[:n]
    top_movies_ids = [m for m, _ in  top_n]
    return movies[movies["movie_id"].isin(top_movies_ids)][["movie_id", "movie_title"]]
recommend_movies(196,n=10)

,movie_id,movie_title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)
5,6,Shanghai Triad (Yao a yao yao dao waipo qiao) ...
6,7,Twelve Monkeys (1995)
7,8,Babe (1995)
8,9,Dead Man Walking (1995)
9,10,Richard III (1995)


In [39]:
from surprise import SVD,Dataset,Reader,accuracy
from surprise.model_selection import cross_validate

In [40]:
# Load using Surprise's Reader (same u1.base/u1.test files you already have)

reader = Reader(line_format="user item rating timestamp", sep="\t")

train_data = Dataset.load_from_file("../data/raw/u1.base",reader=reader)


In [41]:
# Train SVD
trainset = train_data.build_full_trainset()

algorithm = SVD(random_state=42)
algorithm.fit(trainset)

In [42]:
# Evaluate on u1.test — need to load it as raw tuples
test_df = pd.read_csv("../data/raw/u1.test",sep="\t", names=['user_id',"movie_id",'rating','timestamp'])

testset = list(zip(test_df["user_id"],test_df["movie_id"],test_df["rating"]))
predictions = algorithm.test(testset)


accuracy.rmse(predictions)
accuracy.mae(predictions)

RMSE: 1.1537
MAE:  0.9680


np.float64(0.968048775)